# Exploratory Data Analysis

First look at the loan portfolio before building the scorecard.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/loan_portfolio.csv", parse_dates=["origination_date", "default_date"])

## 1. How many rows and columns are there, and which columns have missing values?

In [ ]:
# Dataset dimensions and missing values
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

missing_values = df.isna().sum()
display(missing_values[missing_values > 0].sort_values(ascending=False).to_frame("missing_count"))

## 2. What share of loans ever defaulted?

In [ ]:
# Share of loans that ever defaulted
ever_default_rate = df["defaulted"].mean()
print(f"Lifetime default rate: {ever_default_rate:.2%}")

## 3. How many loans were originated each year?

In [ ]:
# Number of loans originated in each calendar year
loans_per_year = (
    df.assign(origination_year=df["origination_date"].dt.year)
      .groupby("origination_year")
      .size()
      .rename("loan_count")
)

display(loans_per_year.to_frame())

## 4. What is the 12-month default rate by origination year? (bar chart)

In [ ]:
# Define a 12-month default: default_date must fall within 12 months
# after the loan's origination date.
twelve_month_cutoff = df["origination_date"] + pd.DateOffset(months=12)

df["bad_12m"] = (
    df["default_date"].notna()
    & (df["default_date"] <= twelve_month_cutoff)
).astype(int)

df["origination_year"] = df["origination_date"].dt.year

default_rate_by_year = (
    df.groupby("origination_year")
      .agg(
          loan_count=("loan_id", "size"),
          bad_12m_count=("bad_12m", "sum"),
          default_rate_12m=("bad_12m", "mean")
      )
)

display(default_rate_by_year.style.format({"default_rate_12m": "{:.2%}"}))

ax = (default_rate_by_year["default_rate_12m"] * 100).plot(
    kind="bar", figsize=(10, 5), color="steelblue", edgecolor="black"
)
ax.set_title("12-Month Default Rate by Origination Year")
ax.set_xlabel("Origination year")
ax.set_ylabel("12-month default rate (%)")
ax.axhline(2, color="darkorange", linestyle="--", label="2% reference")
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Which years look strange, and why might that be?

In [ ]:
# Flag years materially above the approximate 2% baseline
display(
    default_rate_by_year.loc[
        default_rate_by_year["default_rate_12m"] >= 0.04,
        ["loan_count", "bad_12m_count", "default_rate_12m"]
    ].style.format({"default_rate_12m": "{:.2%}"})
)

print(
    "The 2019 and 2020 cohorts stand out. A possible explanation is "
    "a cohort-specific increase in borrower risk or sector mix; this "
    "should be investigated rather than assumed to be a causal effect."
)

## 6. What is the 12-month default rate by initial_rating?

In [ ]:
# Compare 12-month default rates across initial credit ratings
rating_order = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC"]

default_rate_by_rating = (
    df.groupby("initial_rating")
      .agg(
          loan_count=("loan_id", "size"),
          bad_12m_count=("bad_12m", "sum"),
          default_rate_12m=("bad_12m", "mean")
      )
      .reindex(rating_order)
)

display(default_rate_by_rating.style.format({"default_rate_12m": "{:.2%}"}))

ax = (default_rate_by_rating["default_rate_12m"] * 100).plot(
    kind="bar", figsize=(9, 5), color="slateblue", edgecolor="black"
)
ax.set_title("12-Month Default Rate by Initial Rating")
ax.set_xlabel("Initial rating (best to worst)")
ax.set_ylabel("12-month default rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. What is the 12-month default rate by sector?

In [ ]:
# Compare 12-month default rates across sectors
default_rate_by_sector = (
    df.groupby("sector")
      .agg(
          loan_count=("loan_id", "size"),
          bad_12m_count=("bad_12m", "sum"),
          default_rate_12m=("bad_12m", "mean")
      )
      .sort_values("default_rate_12m", ascending=False)
)

display(default_rate_by_sector.style.format({"default_rate_12m": "{:.2%}"}))

ax = (default_rate_by_sector["default_rate_12m"] * 100).plot(
    kind="bar", figsize=(11, 5), color="seagreen", edgecolor="black"
)
ax.set_title("12-Month Default Rate by Sector")
ax.set_xlabel("Sector")
ax.set_ylabel("12-month default rate (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## Findings

- The dataset contains 50,000 loans and 24 columns; 43,050 loans have no `default_date`, which is expected for loans without a recorded default.
- The lifetime default rate is 13.9%, but the 12-month rate is much lower in most origination cohorts, generally around 2.1%–2.3%.
- The 2019 cohort is unusually high at approximately 12.33%, and 2020 is also elevated at approximately 5.75%; a change in cohort risk or sector mix is one possible explanation that needs further investigation.
- Default rates generally increase as the initial rating worsens, especially for B and CCC, although the ordering is not perfectly monotonic (AA is slightly below AAA).
- Sector rates are relatively close, approximately 3.50%–3.92%; these pooled rates are influenced by the mix of origination years.